# ISIC 2024 — Multimodal Skin Cancer Detection Demo

This notebook demonstrates the end-to-end pipeline:
1. Load the trained multimodal model (EfficientNet + Tabular MLP)
2. Prepare sample data (images + tabular features)
3. Run inference and show predictions
4. Display evaluation metrics and key ablation study results
5. Visualize feature importances and model analysis

**Prerequisites:** Run `python create_sample_data.py` first to generate the sample dataset,  
or use the full dataset by adjusting the paths below.

## 1. Setup and Imports

In [ ]:
import os
import json
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

import torch
import torch.nn as nn
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay

# Import model and utilities from our training code
from train import (
    ISICMultimodalModel, GeM,
    engineer_tabular_features, TABULAR_FEATURE_COLS,
    build_tabular_tensor, compute_pauc, CONFIG,
    NUM_COLS, NEW_NUM_COLS,
)

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

## 2. Load Sample Data

We use a small 50-sample subset (20 positive, 30 negative) created by `create_sample_data.py`.  
To use the full dataset, change `METADATA_PATH` and `IMAGE_DIR` below.

In [ ]:
# --- Paths (change these to use full data) ---
USE_SAMPLE = True

if USE_SAMPLE:
    METADATA_PATH = Path("data/sample/sample_metadata.csv")
    IMAGE_DIR = Path("data/sample/images")
else:
    METADATA_PATH = Path("data/train-metadata.csv")
    IMAGE_DIR = Path("data/train-image/image")

MODEL_PATH = Path("output/final_model.pth")

print(f"Metadata: {METADATA_PATH}")
print(f"Images:   {IMAGE_DIR}")
print(f"Model:    {MODEL_PATH}")
print(f"Metadata exists: {METADATA_PATH.exists()}")
print(f"Model exists:    {MODEL_PATH.exists()}")

## 3. Feature Engineering

Apply the same 75 engineered tabular features used during training.

In [ ]:
df = engineer_tabular_features(METADATA_PATH)
df["file_path"] = df["isic_id"].apply(lambda x: str(IMAGE_DIR / f"{x}.jpg"))
df = df[df["file_path"].apply(os.path.isfile)].reset_index(drop=True)

print(f"Samples: {len(df)}")
print(f"Positive: {df['target'].sum()}  |  Negative: {(df['target'] == 0).sum()}")
df[["isic_id", "target", "age_approx", "sex", "anatom_site_general", "clin_size_long_diam_mm"]].head(10)

## 4. Visualize Sample Images

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 7))

# Show 5 positive and 5 negative samples
pos_samples = df[df["target"] == 1].head(5)
neg_samples = df[df["target"] == 0].head(5)

for i, (_, row) in enumerate(pos_samples.iterrows()):
    img = cv2.imread(row["file_path"])
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    axes[0][i].imshow(img)
    axes[0][i].set_title(f"MALIGNANT\n{row['isic_id']}", fontsize=9, color="red", fontweight="bold")
    axes[0][i].axis("off")

for i, (_, row) in enumerate(neg_samples.iterrows()):
    img = cv2.imread(row["file_path"])
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    axes[1][i].imshow(img)
    axes[1][i].set_title(f"BENIGN\n{row['isic_id']}", fontsize=9, color="green", fontweight="bold")
    axes[1][i].axis("off")

plt.suptitle("Sample Lesion Images", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 5. Load Trained Model

In [ ]:
n_tab_features = len(TABULAR_FEATURE_COLS)
print(f"Tabular features: {n_tab_features}")

model = ISICMultimodalModel(
    model_name=CONFIG["model_name"],
    n_tabular_features=n_tab_features,
    pretrained=False,  # we'll load our own weights
)

state = torch.load(MODEL_PATH, map_location=DEVICE, weights_only=True)
model.load_state_dict(state)
model.to(DEVICE)
model.eval()

total_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded — {total_params:,} parameters")

## 6. Run Inference on Sample Data

In [ ]:
# Prepare tabular features
tab_array = build_tabular_tensor(df, TABULAR_FEATURE_COLS)
tab_mean = tab_array.mean(axis=0)
tab_std = tab_array.std(axis=0) + 1e-8
tab_array = (tab_array - tab_mean) / tab_std

# Image transform (same as validation)
transform = A.Compose([
    A.Resize(CONFIG["img_size"], CONFIG["img_size"]),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225], max_pixel_value=255.0),
    ToTensorV2(),
])

# Run inference
predictions = []

with torch.inference_mode():
    for i in range(len(df)):
        # Load and transform image
        img = cv2.imread(df["file_path"].iloc[i])
        if img is None:
            img = np.zeros((CONFIG["img_size"], CONFIG["img_size"], 3), dtype=np.uint8)
        else:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img_tensor = transform(image=img)["image"].unsqueeze(0).to(DEVICE)

        # Tabular tensor
        tab_tensor = torch.tensor(tab_array[i], dtype=torch.float32).unsqueeze(0).to(DEVICE)

        # Predict
        pred = model(img_tensor, tab_tensor).item()
        predictions.append(pred)

df["prediction"] = predictions
print(f"Inference complete on {len(df)} samples.")
df[["isic_id", "target", "prediction"]].head(10)

## 7. Evaluate Predictions

In [ ]:
y_true = df["target"].values
y_pred = df["prediction"].values

# AUC
if len(np.unique(y_true)) > 1:
    auc = roc_auc_score(y_true, y_pred)
    pauc = compute_pauc(y_true, y_pred)
    print(f"AUC-ROC:          {auc:.4f}")
    print(f"pAUC @ 80% TPR:   {pauc:.4f}")
else:
    print("Cannot compute AUC — only one class present in sample.")

# ROC curve
if len(np.unique(y_true)) > 1:
    fpr, tpr, _ = roc_curve(y_true, y_pred)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(fpr, tpr, color="#4C72B0", linewidth=2)
    axes[0].plot([0, 1], [0, 1], "--", color="gray", alpha=0.5)
    axes[0].fill_between(fpr, tpr, alpha=0.1, color="#4C72B0")
    axes[0].set_xlabel("False Positive Rate")
    axes[0].set_ylabel("True Positive Rate")
    axes[0].set_title(f"ROC Curve (AUC = {auc:.4f})")
    axes[0].grid(True, alpha=0.3)

    # Prediction distribution
    axes[1].hist(y_pred[y_true == 0], bins=20, alpha=0.6, label="Benign", color="green", edgecolor="white")
    axes[1].hist(y_pred[y_true == 1], bins=20, alpha=0.6, label="Malignant", color="red", edgecolor="white")
    axes[1].set_xlabel("Predicted Probability")
    axes[1].set_ylabel("Count")
    axes[1].set_title("Prediction Score Distribution")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

## 8. Ablation Study Results

We trained 5 model variants to isolate the contribution of each component.  
Results below are loaded from the saved JSON files produced by `train.py` and `ablations.py`.

In [ ]:
# Load saved results
all_results = {}

baseline_path = Path("output/baseline_results.json")
if baseline_path.exists():
    with open(baseline_path) as f:
        all_results["Baseline"] = json.load(f)

ablation_path = Path("output/ablations/ablation_results.json")
if ablation_path.exists():
    with open(ablation_path) as f:
        all_results.update(json.load(f))

if all_results:
    results_df = pd.DataFrame([
        {
            "Experiment": name,
            "Image Branch": r.get("image_branch", "-"),
            "Tabular Branch": r.get("tabular_branch", "-"),
            "Pooling": r.get("pooling", "-"),
            "Test AUC": f"{r['test_auc']:.4f}",
            "Test pAUC": f"{r['test_pauc']:.4f}",
        }
        for name, r in all_results.items()
    ])
    display(results_df.style.set_caption("Ablation Study Results").hide(axis='index'))
else:
    print("No saved results found. Run train.py and ablations.py first.")

In [ ]:
# Bar chart comparison
if all_results:
    names = list(all_results.keys())
    aucs = [all_results[n]["test_auc"] for n in names]
    paucs = [all_results[n]["test_pauc"] for n in names]

    x = np.arange(len(names))
    width = 0.32

    fig, ax = plt.subplots(figsize=(12, 5))
    bars1 = ax.bar(x - width/2, aucs, width, label="AUC", color="#4C72B0", edgecolor="white")
    bars2 = ax.bar(x + width/2, paucs, width, label="pAUC @ 80% TPR", color="#DD8452", edgecolor="white")

    ax.set_ylabel("Score")
    ax.set_title("Ablation Study — Test Set Performance", fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels(names, rotation=15, ha="right")
    ax.legend()
    ax.grid(True, axis="y", alpha=0.3)

    for bar in bars1:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f"{bar.get_height():.4f}", ha="center", fontsize=8, fontweight="bold")
    for bar in bars2:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f"{bar.get_height():.4f}", ha="center", fontsize=8, fontweight="bold")

    plt.tight_layout()
    plt.show()

## 9. Training Curves

In [ ]:
all_histories = {}

bh_path = Path("output/baseline_history.json")
if bh_path.exists():
    with open(bh_path) as f:
        all_histories["Baseline"] = json.load(f)

ah_path = Path("output/ablations/ablation_histories.json")
if ah_path.exists():
    with open(ah_path) as f:
        all_histories.update(json.load(f))

if all_histories:
    colors = {"Baseline": "#2ecc71", "Ablation_A": "#3498db", "Ablation_B": "#e74c3c",
              "Ablation_C": "#9b59b6", "Ablation_D": "#f39c12"}

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    metrics = [("val_loss", "Validation Loss"), ("val_auc", "Validation AUC"), ("val_pauc", "Validation pAUC")]

    for ax, (key, title) in zip(axes, metrics):
        for name, hist in all_histories.items():
            if key in hist:
                epochs = range(1, len(hist[key]) + 1)
                ax.plot(epochs, hist[key], marker="o", markersize=3, linewidth=1.5,
                        label=name, color=colors.get(name))
        ax.set_title(title, fontweight="bold")
        ax.set_xlabel("Epoch")
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
else:
    print("No training histories found.")

## 10. Feature Importance Analysis

We examine the first linear layer of the tabular MLP to see which features  
the model learned to weight most heavily.

In [ ]:
# Extract first-layer weights: shape (256, 75)
W_tab = model.tab_mlp[0].weight.detach().cpu().numpy()
importance = np.linalg.norm(W_tab, axis=0)  # L2 norm per feature

feat_names = [c.replace("tbp_lv_", "").replace("_", " ") for c in TABULAR_FEATURE_COLS]

# Top 20
order = np.argsort(importance)[::-1][:20]

fig, ax = plt.subplots(figsize=(9, 7))
y_pos = np.arange(len(order))
ax.barh(y_pos, importance[order][::-1], color="#4C72B0", edgecolor="white")
ax.set_yticks(y_pos)
ax.set_yticklabels([feat_names[i] for i in order][::-1], fontsize=9)
ax.set_xlabel("Weight L2 Norm")
ax.set_title("Top 20 Tabular Feature Importances\n(First MLP Layer)", fontweight="bold")
ax.grid(True, axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

## 11. First Conv Layer Analysis ($W^T W$ Gram Matrix)

The first convolutional layer of EfficientNet-B0 has 32 filters of size 3×3×3.  
Computing $W^T W$ reveals learned correlations across spatial-color dimensions.

In [ ]:
conv_weight = model.backbone.conv_stem.weight.detach().cpu().numpy()  # (32, 3, 3, 3)
out_ch, in_ch, kH, kW = conv_weight.shape
print(f"First conv layer shape: {conv_weight.shape}")

W = conv_weight.reshape(out_ch, -1)  # (32, 27)
G = W.T @ W                           # (27, 27)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Gram matrix
vmax = np.abs(G).max()
norm = TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)
im = axes[0].imshow(G, cmap="RdBu_r", norm=norm, aspect="equal")
axes[0].set_title("Gram Matrix $W^T W$ (27×27)\nFirst Conv Layer", fontweight="bold")

ch_names = ["R", "G", "B"]
tick_labels = [f"{ch_names[c]}({ky},{kx})" for c in range(in_ch) for ky in range(kH) for kx in range(kW)]
axes[0].set_xticks(range(len(tick_labels)))
axes[0].set_xticklabels(tick_labels, rotation=90, fontsize=5)
axes[0].set_yticks(range(len(tick_labels)))
axes[0].set_yticklabels(tick_labels, fontsize=5)
fig.colorbar(im, ax=axes[0], fraction=0.046, pad=0.04)

# Filter visualization
grid_rows, grid_cols = 4, 8
filter_img = np.zeros((grid_rows * (kH + 1) - 1, grid_cols * (kW + 1) - 1, 3))

for i in range(out_ch):
    r, c = divmod(i, grid_cols)
    filt = conv_weight[i].transpose(1, 2, 0)
    filt = (filt - filt.min()) / (filt.max() - filt.min() + 1e-8)
    y0, x0 = r * (kH + 1), c * (kW + 1)
    filter_img[y0:y0+kH, x0:x0+kW, :] = filt

axes[1].imshow(filter_img)
axes[1].set_title(f"Learned First Conv Filters ({out_ch}× {kH}×{kW} RGB)", fontweight="bold")
axes[1].axis("off")

plt.tight_layout()
plt.show()

## 12. Model Architecture Summary

In [ ]:
print("=" * 60)
print("MODEL ARCHITECTURE")
print("=" * 60)
print()
print("Image Branch:")
print(f"  Backbone:  EfficientNet-B0 (pretrained, fine-tuned)")
print(f"  Pooling:   GeM (learned p)")
print(f"  Output:    1280-d embedding")
print()
print("Tabular Branch:")
print(f"  Input:     {n_tab_features} features")
print(f"  Layers:    Linear(75→256) → BN → ReLU → Drop(0.3)")
print(f"             Linear(256→128) → BN → ReLU → Drop(0.2)")
print(f"  Output:    128-d embedding")
print()
print("Fusion Head:")
print(f"  Input:     1280 + 128 = 1408-d")
print(f"  Layers:    Linear(1408→256) → BN → ReLU → Drop(0.3)")
print(f"             Linear(256→1) → Sigmoid")
print()
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters:     {total:,}")
print(f"Trainable parameters: {trainable:,}")

## 13. Conclusion

**Key Findings:**

1. **Multimodal fusion outperforms single modalities.** The best model (AvgPool variant) achieves 0.939 AUC, surpassing both tabular-only (0.928) and image-only (0.906).

2. **Tabular features are surprisingly powerful.** The MLP on engineered features alone outperforms the EfficientNet CNN, consistent with competition leaderboard trends.

3. **Fine-tuning is essential.** The frozen-backbone variant (0.882) performs worst, confirming that ImageNet features must be adapted for dermoscopy.

4. **Pooling choice matters.** AvgPool outperforms GeM in the fusion setting, likely due to reduced overfitting with limited positive samples.

5. **Feature importance analysis** reveals the model relies most on lesion geometry and color contrast features, aligning with clinical dermatology knowledge (ABCDE criteria).